# Phase 1 (vérité terrain) — Reconnaître JEMBÉ vs DUNUN

On utilise le corpus **IEMP Malian Jembe** (Polak) : des pistes **isolées** par instrument (1 micro/tambour) + les onsets de chaque frappe. C'est notre **vérité terrain**.

Idée : on n'a plus à *deviner*. On **apprend** le timbre du jembé et du dunun sur des exemples étiquetés, on entraîne un classifieur, puis on l'**applique au mix YouTube** (qui, lui, mélange tout).

**Plan :** 1) jeu étiqueté depuis les pistes isolées → 2) écoute de vérif → 3) classifieur jembé/dunun → 4) carte UMAP étiquetée → 5) application au Woloso YouTube → 6) écoute des prédictions.

> Le corpus = style Bamako : **jembé + dunun uniquement** (pas de cloche). Le modèle saura donc séparer jembé/dunun ; une éventuelle cloche dans les vidéos sera mal classée (à traiter plus tard en non-supervisé).

## 0. Imports, config & fonctions communes

Mêmes features et même détection d'onsets pour l'entraînement (pistes isolées) ET l'inférence (mix YouTube) — sinon le modèle compare des choux et des carottes.

In [ ]:
import os, re, glob, warnings
import numpy as np, pandas as pd
import librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import umap

warnings.filterwarnings('ignore')

SR  = 22050
WIN = 0.080
CORP = 'corpus/MJ_Woloso'
OUT  = 'output'; os.makedirs(OUT, exist_ok=True)

FEAT = ['centroid','bandwidth','rolloff','flatness','zcr','rms'] + [f'mfcc{i}' for i in range(13)]
INSTR_COLORS = {'jembe':'#d63bd6', 'dunun':'#2b6cff'}

def stroke_features(seg):
    mfcc = librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=13).mean(axis=1)
    f = {'centroid':  librosa.feature.spectral_centroid(y=seg, sr=SR).mean(),
         'bandwidth': librosa.feature.spectral_bandwidth(y=seg, sr=SR).mean(),
         'rolloff':   librosa.feature.spectral_rolloff(y=seg, sr=SR).mean(),
         'flatness':  librosa.feature.spectral_flatness(y=seg).mean(),
         'zcr':       librosa.feature.zero_crossing_rate(seg).mean(),
         'rms':       librosa.feature.rms(y=seg).mean()}
    f.update({f'mfcc{i}': v for i, v in enumerate(mfcc)})
    return f

def detect_onsets(y, delta=0.2, wait=10):
    fr = librosa.onset.onset_detect(y=y, sr=SR, backtrack=True, units='frames',
                                    pre_max=20, post_max=20, pre_avg=100, post_avg=100,
                                    delta=delta, wait=wait)
    return librosa.frames_to_time(fr, sr=SR)

print('Config OK')

## 1. Construire le jeu ÉTIQUETÉ depuis les pistes isolées

Pour chaque prise et chaque instrument isolé (J1/J2 = jembé, D1/D2 = dunun), on lit les **onsets de référence** fournis (`Onsets_Raw_*.csv`), on découpe une fenêtre autour de chaque frappe et on extrait les features. Le label est connu = nom de la piste.

In [ ]:
# code piste -> (nom dans le CSV d'onsets, label instrument)
IMAP = {'J1':('Jembe1','jembe'), 'J2':('Jembe2','jembe'),
        'D1':('Dundun1','dunun'), 'D2':('Dundun2','dunun')}

rows, AUDIO_ISO = [], {}
wl = int(WIN*SR)
for wav in sorted(glob.glob(f'{CORP}/MJ_Woloso_*_*.wav')):
    m = re.match(r'MJ_Woloso_(\d+)_(J1|J2|D1|D2)\.wav', os.path.basename(wav))
    if not m:
        continue
    take, code = int(m.group(1)), m.group(2)
    onset_name, label = IMAP[code]
    csv = f'{CORP}/annotations/MJ_Woloso_{take}_Onsets_Raw_{onset_name}.csv'
    if not os.path.exists(csv):
        print('onsets manquants:', csv); continue
    y, _ = librosa.load(wav, sr=SR, mono=True)
    AUDIO_ISO[(take, code)] = y
    times = pd.read_csv(csv)['Time'].values
    for t in times:
        s = int(t*SR); seg = y[s:s+wl]
        if len(seg) < wl//2:
            continue
        row = {'take':take, 'code':code, 'label':label, 'time':float(t)}
        row.update(stroke_features(seg)); rows.append(row)

df = pd.DataFrame(rows)
print('Frappes étiquetées :', len(df))
print(df.groupby(['label','code']).size())
print('\nPar prise :'); print(df.groupby(['take','label']).size().unstack(fill_value=0))

## 2. Vérifier à l'oreille
On enchaîne quelques frappes de chaque instrument (depuis les pistes isolées) pour confirmer que les labels sont bons.

In [ ]:
def ecouter_iso(label, n=14, dur=0.25):
    sub = df[df['label']==label].sample(min(n,(df['label']==label).sum()), random_state=1)
    clips = []
    for _, r in sub.iterrows():
        y = AUDIO_ISO[(r['take'], r['code'])]; s = int(r['time']*SR)
        clips.append(y[s:s+int(dur*SR)])
    print(f'>>> {label}'); display(Audio(np.concatenate(clips), rate=SR))

for lab in ['dunun','jembe']:
    ecouter_iso(lab)

## 3. Entraîner le classifieur jembé / dunun

On **sépare par prise** (train = prises 1–4, test = prise 5) pour mesurer une vraie généralisation (pas juste de la mémorisation).

In [ ]:
train = df[df['take'] < 5]
test  = df[df['take'] == 5]
clf = Pipeline([('sc', StandardScaler()),
                ('rf', RandomForestClassifier(n_estimators=300, random_state=0,
                                              class_weight='balanced'))])
clf.fit(train[FEAT], train['label'])
pred = clf.predict(test[FEAT])

print('Test = prise 5 (', len(test), 'frappes )\n')
print(classification_report(test['label'], pred, digits=3))
ConfusionMatrixDisplay.from_predictions(test['label'], pred, cmap='Purples')
plt.title('Matrice de confusion (prise 5)'); plt.show()

In [ ]:
# quelles features comptent le plus ?
imp = pd.Series(clf.named_steps['rf'].feature_importances_, index=FEAT).sort_values()
imp.tail(12).plot.barh(figsize=(8,5), color='#d63bd6')
plt.title('Importance des features (jembé vs dunun)'); plt.tight_layout(); plt.show()

## 4. Carte UMAP étiquetée (vérité terrain)
La carte façon « oiseaux », mais ici les couleurs sont des **vrais labels**, pas des suppositions.

In [ ]:
Xs = StandardScaler().fit_transform(df[FEAT])
emb = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(Xs)
df['ux'], df['uy'] = emb[:,0], emb[:,1]

plt.figure(figsize=(10,8), facecolor='#0d0d0d'); ax=plt.gca(); ax.set_facecolor('#0d0d0d')
for lab,c in INSTR_COLORS.items():
    m = df['label']==lab
    plt.scatter(df.loc[m,'ux'], df.loc[m,'uy'], s=6, c=c, alpha=.5, label=lab)
plt.legend(markerscale=3, labelcolor='white', facecolor='#1a1a1a')
ax.tick_params(colors='#555'); plt.title('Frappes isolées jembé vs dunun (UMAP, étiqueté)', color='white')
plt.savefig(f'{OUT}/umap_groundtruth.png', dpi=140, facecolor='#0d0d0d'); plt.show()

## 5. Appliquer au Woloso YOUTUBE — test de robustesse

On applique le modèle (appris sur des pistes isolées *propres*) au **mix YouTube réel**.

> ⚠️ **À surveiller** : si les prédictions s'effondrent sur une seule classe (ex. ~100 % jembé, ~0 % dunun), ce **n'est pas** un succès — c'est le signe d'un **décalage de domaine** (pistes isolées micro-peau ≠ mix capté en salle) et du **chevauchement** des instruments. On le diagnostique en section 7.

In [ ]:
y_yt, _ = librosa.load('audio/Woloso.wav', sr=SR, mono=True)
on = detect_onsets(y_yt)
rows = []
for t in on:
    s = int(t*SR); seg = y_yt[s:s+wl]
    if len(seg) < wl//2:
        continue
    r = {'time':float(t)}; r.update(stroke_features(seg)); rows.append(r)
yt = pd.DataFrame(rows)
yt['pred'] = clf.predict(yt[FEAT])
print(len(yt), 'frappes détectées sur le mix YouTube')
print(yt['pred'].value_counts(normalize=True).round(3))

plt.figure(figsize=(14,4))
for lab,c in INSTR_COLORS.items():
    m = yt['pred']==lab
    plt.scatter(yt.loc[m,'time'], yt.loc[m,'centroid'], s=14, c=c, alpha=.6, label=lab)
plt.legend(); plt.xlabel('temps (s)'); plt.ylabel('centroïde (Hz)')
plt.title('Woloso YouTube — frappes classées par instrument prédit'); plt.xlim(30,55); plt.show()

## 6. Écouter les prédictions sur le mix
Le vrai test : est-ce que les frappes prédites « dunun » sonnent graves, et les « jembé » plus claires ?

In [ ]:
def ecouter_yt(label, n=16, dur=0.25):
    sub = yt[yt['pred']==label].sample(min(n,(yt['pred']==label).sum()), random_state=2)
    clips = [y_yt[int(r['time']*SR):int(r['time']*SR)+int(dur*SR)] for _,r in sub.iterrows()]
    print(f'>>> prédit {label}'); display(Audio(np.concatenate(clips), rate=SR))

for lab in ['dunun','jembe']:
    ecouter_yt(lab)

## 7. Diagnostic : pourquoi ça s'effondre sur le mix ?

Deux causes possibles à l'effondrement vers « jembé » :
1. **Chevauchement** — au moment d'un coup de dunun, un jembé sonne souvent en même temps → la fenêtre devient large bande → classée jembé.
2. **Décalage de domaine** — corpus = micro collé sur la peau (son très sec) ; YouTube = capté en salle (réverb, EQ, autre accordage).

Pour isoler l'effet du **seul chevauchement**, on fabrique un **mix synthétique** à partir des pistes isolées (qu'on sait synchronisées) et on reclasse les onsets connus du dunun. Si le % dunun chute déjà ici, c'est le chevauchement ; la chute supplémentaire sur le vrai YouTube, c'est le domaine.

In [ ]:
# Mix SYNTHÉTIQUE depuis les pistes isolées (prise 5) : isole l'effet du SEUL chevauchement
wl = int(WIN*SR)
d1, _ = librosa.load(f'{CORP}/MJ_Woloso_5_D1.wav', sr=SR, mono=True)
j1, _ = librosa.load(f'{CORP}/MJ_Woloso_5_J1.wav', sr=SR, mono=True)
n = min(len(d1), len(j1)); d1, j1 = d1[:n], j1[:n]
mix = d1 + j1
dt = pd.read_csv(f'{CORP}/annotations/MJ_Woloso_5_Onsets_Raw_Dundun1.csv')['Time'].values

def pct_dunun(sig, times):
    X = [stroke_features(sig[int(t*SR):int(t*SR)+wl]) for t in times
         if len(sig[int(t*SR):int(t*SR)+wl]) >= wl//2]
    return (clf.predict(pd.DataFrame(X)[FEAT]) == 'dunun').mean()

print('Aux onsets connus du DUNUN (prise 5) :')
print(f'  piste dunun ISOLÉE          : {pct_dunun(d1, dt)*100:5.1f}% classés dunun')
print(f'  mix synthétique dunun+jembé : {pct_dunun(mix, dt)*100:5.1f}% classés dunun')
print('\nÉcoute le mix synthétique (dunun + jembé) :')
display(Audio(mix, rate=SR))

## 8. Bilan & suite (corrigée)

| Situation | % dunun retrouvé |
|---|---|
| isolé → isolé (test prise 5) | **~99,8 %** ✅ |
| chevauchement seul (mix synthétique) | **~69 %** |
| mix YouTube réel | **~0,5 %** ❌ |

→ Classer chaque frappe du **mix brut** ne marche pas pour les instruments : chevauchement + décalage de domaine effondrent le dunun. Le dunun se reconnaît surtout par sa **basse fréquence**, le seul indice qui traverse les domaines.

**Prochaine étape (corrigée) :**
- **Séparer d'abord** : NMF ou filtrage en **bande grave (<200 Hz)** pour repérer le dunun *sur le mix*, puis classer les stems séparés avec ce modèle (qui reste excellent en isolé).
- Garder ce classifieur comme **brique validée** (étiquetage de stems, et utile pour la Phase 2).
- **Phase 2** : bass/tone/slap sur la voix jembé (la piste `J1` isolée du corpus = idéale, propre).
- **Phase 3** : cycle rythmique via `Onsets_Selected` (positions métriques) du Woloso.

In [ ]:
yt.to_csv(f'{OUT}/youtube_woloso_predictions.csv', index=False)
print('Sauvegardé ->', f'{OUT}/youtube_woloso_predictions.csv', '|', len(yt), 'frappes')